# Lesson 25 · Walking downhill blindfolded, following the negative gradient to the valley floor — gradient descent (GD) and the weight-update rule

**Goal**: to make a function smaller, take a small step in the **opposite direction** to the gradient, and repeat.

`new position = old position − learning rate (lr) × gradient`

**Why it matters for Aurora**: every call to `optimizer.step()` in `aurora.train` runs exactly this one-line update rule; set the learning rate wrong, and the convergence curve will oscillate on either side of the minimum or stall for a long time.

← **Previous lesson** · [L24 · The chain rule](L24_chain_rule.ipynb)

> Last lesson covered **the chain rule**: differentiating nested functions, the maths behind backpropagation.
> This lesson explores **gradient descent**.

## This lesson's story: walking downhill blindfolded

You're blindfolded, standing somewhere on a hillside, and your goal is to reach the valley.

What you can do: feel the slope where you're standing with your feet (= the gradient), then take one step in the steepest downhill direction (= the negative gradient). Repeat, and you get closer and closer to the valley floor.

That is **gradient descent**:

```
x ← x - lr · ∇f(x)
```

- `x`: the current position (the weight vector)
- `∇f(x)`: the current slope (the gradient)
- `lr`: the step size (the learning rate)

Three core questions:
1. **Why the "negative" gradient?** The gradient points uphill; walking the opposite way is downhill.
2. **What happens if lr is too large?** With steps that are too big, you may overshoot the valley floor and bounce to the other side (oscillation/divergence).
3. **When do you stop?** When the gradient is close to zero (∇f(x) ≈ 0) — **but at that point you may be at any of several kinds of place**:
   - **A valley floor (local minimum)**: every direction around you is uphill. This is the ideal place to stop.
   - **A saddle point**: some directions go up and some go down (like a saddle between two peaks). The gradient is zero, but it isn't a minimum.
   - **A plateau**: the gradient is very small but not exactly zero, and the function value changes slowly.

   In deep learning, **gradient descent in high-dimensional spaces is more likely to get stuck at saddle points than at local minima** — which is why L58 later uses more advanced optimizers such as momentum. Don't worry about it in this lesson: a simple quadratic like f(x)=(x−3)² has no saddle points.

In this lesson you'll implement a single update, `gd_step(x, grad_value, lr)`, and check that
after 50 steps, f(x)=(x-3)² converges from x=0 to x≈3.

## 1. One step by hand: f(x) = (x−3)²

**Analytical gradient**: f'(x) = 2(x−3)

**Starting point**: x₀ = 0, learning rate lr = 0.1

**First update**:
```
x₁ = x₀ - lr · f'(x₀)
   = 0 - 0.1 · f'(0)
   = 0 - 0.1 · 2·(0-3)
   = 0 - 0.1 · (-6)
   = 0.6
```

**Second step**:
```
x₂ = x₁ - 0.1 · f'(x₁)
   = 0.6 - 0.1 · 2·(0.6-3)
   = 0.6 - 0.1 · (-4.8)
   = 1.08
```

After each step x is closer to the minimizer x*=3, and the step size shrinks geometrically with common ratio `(1 - 2·lr)` = 0.8.

---

### 🔬 Why does the error get multiplied by (1 - 2·lr) each step? A detailed derivation

This isn't magic — let's derive exactly where this ratio comes from.

**Define the error** e_k = x_k - 3 (the distance from the current position to the optimum, 3)

**The maths of one iteration**:
```
x_{k+1} = x_k - lr · f'(x_k)
        = x_k - lr · 2(x_k - 3)                  [substitute f'(x)=2(x-3)]
        = x_k - 2·lr·x_k + 2·lr·3
        = x_k(1 - 2·lr) + 6·lr
```

**Rewrite in terms of the error**. Note that x_k = e_k + 3 (the error plus the optimum); substitute into the line above:
```
x_{k+1} = (e_k + 3)(1 - 2·lr) + 6·lr
        = e_k(1 - 2·lr) + 3(1 - 2·lr) + 6·lr
        = e_k(1 - 2·lr) + 3 - 6·lr + 6·lr
        = e_k(1 - 2·lr) + 3
```

So:
```
e_{k+1} = x_{k+1} - 3 = (1 - 2·lr)·e_k
```

**What does this tell us?**
- At every step, the error is multiplied by the same number (1 - 2·lr) (called the **common ratio**)
- If |ratio| < 1, the error decays exponentially → it converges to 0 (i.e. x → 3)
- If |ratio| ≥ 1, the error doesn't shrink, and the sequence can't approach the optimum

**Checking the example f(x)=(x-3)², lr=0.1**:
- ratio = 1 - 2×0.1 = 0.8
- e_0 = 0 - 3 = -3
- e_1 = 0.8 × (-3) = -2.4, so x_1 = -2.4 + 3 = 0.6 ✓
- e_2 = 0.8 × (-2.4) = -1.92, so x_2 = -1.92 + 3 = 1.08 ✓

---

**Boundary conditions on lr** (for this example, f'(x)=2(x−3), judging convergence by the ratio):
- **0 < lr < 1**: |1 − 2·lr| < 1 → the ratio lies in (−1, 1) → converges (the error shrinks by a factor of |ratio| each step)
- **lr = 0.5**: 1 − 2·0.5 = 0 → the ratio is 0 → e_1 = 0, **the optimum in a single step** (x₁ = 3)
- **lr = 1**: 1 − 2·1 = −1 → ratio −1 → e_k alternates in sign, and x **oscillates forever** between 0 and 6, never reaching 3
- **lr > 1**: |1 − 2·lr| > 1 → ratio magnitude > 1 → the error grows → **divergence**

💡 **Why might this rule be different for other functions?** This analysis rests on the specific form of the derivative f'(x)=2(x−3). For other functions, the coefficient of the derivative (the "2" here) changes, and so does the expression for the ratio. **So "0<lr<1 converges" can't be blindly applied to every function** — you have to derive the lr bounds for the specific f'(x) (here a quadratic, with second derivative 2).

## Into the lab: watching a function through numerical change

Three key variables: `x` (the current position), `grad` (the gradient at that point), and `lr` (the learning rate). Watch how `x` approaches 3 after each iteration, and how `f(x)` converges to 0.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
grad = lambda x: 2*(x-3)      # f'(x)
x = 0.0; lr = 0.1
for step in range(30):
    x = x - lr * grad(x)
print('converged to x =', round(x, 4), ' (the minimum is at x=3)')

## Hands-on: a rate of change is more than a slogan

First compute the gradient and one update at a single point and confirm the numbers are right, then move on to iterating over many steps.

### 📐 Numerical differentiation: approximating the derivative with finite differences

The mathematical definition of the derivative is a limit:
$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

But a computer can't really take a limit; it can only imitate one with a **very small h** (say 1e-3). That's **numerical differentiation**.

There are a few options:
- **Backward difference**: $\frac{f(x) - f(x-h)}{h}$ (looking back)
- **Forward difference**: $\frac{f(x+h) - f(x)}{h}$ (looking ahead)
- **Central difference**: $\frac{f(x+h) - f(x-h)}{2h}$ (looking both ways)

**Why choose the central difference?** Because it measures the change symmetrically on both sides of x, its accuracy is one order higher than a one-sided difference. A Taylor expansion proves it: the central difference's error is O(h²), while the backward/forward difference's error is O(h).

**Do we lose floating-point precision?** We do. If h is too small (say 1e-15), f(x+h) and f(x-h) become almost identical, and the leading digits of their difference are lost (cancellation error). So h has to be a compromise — usually between 1e-4 and 1e-5, close enough to approximate the derivative while avoiding the loss of floating-point precision.

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('approx. slope =', np.round(slopes, 3))
print('theoretical slope 2x =', 2 * xs)


## Code experiment: walk through different positions and watch the slope change

Walk `x` from −3 to 3, print the function value and slope at each point, and see clearly how the sign of the gradient changes on either side of the minimum.

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ Implement a single update `gd_step(x, grad_value, lr)`

**Line of reasoning**:
1. `grad_value` points in the direction in which the function rises fastest; to make the function value fall, subtract it from the current position: `x - something`.
2. The step size is controlled by `lr`: the actual move is `lr * grad_value`, rather than subtracting the whole gradient directly (the gradient's value may be large).
3. Putting it together: `x_new = x - lr * grad_value` — the whole function is a single line, no loop needed.

**Reference input and output**: `f(x)=(x−3)²`, current `x=0`, gradient `2(0−3)=−6`, `lr=0.1` → new `x = 0 − 0.1×(−6) = 0.6` (closer to 3).

> 💡 Need a hint? Once you've finished the exercise, you can compare with the reference implementation in the `solutions/` folder.


### Before you write code, fill in the variable table

Before writing `gd_step`, pin down three things:
- Input: `x` (the current position, a float), `grad_value` (the gradient at that point), `lr` (the learning rate, which controls the step size)
- Key step: **subtract** the move (`lr` times the gradient direction) from the current position to get the new position
- Return: the new position after one step of gradient descent


In [ ]:
def gd_step(x, grad_value, lr):
    # ✏️ TODO: return the updated x
    raise NotImplementedError("Implement gd_step: return the new position, x minus lr times the gradient")


In [ ]:
# Single-step check — implement gd_step first, then run this cell
# Expected: gd_step(0.0, -6.0, 0.1) = 0.0 - 0.1*(-6.0) = 0.6
try:
    result = gd_step(0.0, -6.0, 0.1)
    assert abs(result - 0.6) < 1e-9, f"single-step check failed: expected 0.6, got {result}"
    print('✅ Single-step check passed: gd_step(0.0, -6.0, 0.1) =', result)
except (NotImplementedError, TypeError) as e:
    print('⚠️', e)


In [ ]:
try:
    x = 0.0; lr = 0.1
    for _ in range(50):
        x = gd_step(x, 2*(x-3), lr)
    assert abs(x - 3.0) < 1e-3
    print('✅ Passed: you implemented one step of gradient descent; after iterating, x =', round(x, 4))
except (NotImplementedError, TypeError) as e:
    print('⚠️', e)


## 3. A preview of the real thing: fitting the line y = 2x + 1 (with noise) by gradient descent

(Just run this cell and look at the result — it's a first sketch of Month 2's linear regression)

### From one dimension to many: gradient descent in linear regression

Imagine we want to learn a straight line y = w·x + b from data.

**Setting up the problem**:
- Data points: (X_i, y_i), i = 1, 2, ..., n
- Model prediction: ŷ = w·X + b
- Error: err_i = ŷ_i - y_i (prediction minus true value)
- **Loss function** (measuring all the errors): MSE = (1/n) Σ(err_i)²

Here w and b are the parameters we want to optimize, like the "position x" in the one-dimensional case.

**The partial derivative with respect to w**, ∂(MSE)/∂w:

Apply the chain rule:
```
∂(MSE)/∂w = ∂/∂w [ (1/n) Σ(err_i)² ]
          = (1/n) Σ 2·err_i · ∂(err_i)/∂w
          = (1/n) Σ 2·err_i · X_i                [because err_i = w·X_i + b - y_i, so ∂err_i/∂w = X_i]
          = 2 · (1/n) Σ err_i · X_i
```

**The partial derivative with respect to b**, ∂(MSE)/∂b:
```
∂(MSE)/∂b = (1/n) Σ 2·err_i · ∂(err_i)/∂b
          = (1/n) Σ 2·err_i · 1                  [because ∂err_i/∂b = 1]
          = 2 · (1/n) Σ err_i
```

**The gradient-descent update** (using `np.mean(err * X)` in place of `(1/n) Σ err_i·X_i`):
```python
w -= lr * [∂(MSE)/∂w] = lr * 2 · mean(err * X)
b -= lr * [∂(MSE)/∂b] = lr * 2 · mean(err)
```

🔗 **How it matches the one-dimensional case**: for f(x)=(x−3)², the gradient is 2(x−3), so each step x decreases by lr·2(x−3). Here err plays the role of "x−3" (the amount off from the ideal value), and the gradient also contains a factor of 2. Exactly the same mathematical structure — only now there are two parameters (w, b) instead of one.

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
X = np.linspace(0, 5, 40)
y = 2*X + 1 + rng.normal(0, 0.5, size=X.shape)   # true relationship + noise
w, b, lr = 0.0, 0.0, 0.01
for _ in range(2000):
    pred = w*X + b
    err = pred - y
    w -= lr * np.mean(err * X) * 2     # gradient of the MSE
    b -= lr * np.mean(err) * 2
print(f'learned w={w:.2f} (true value 2), b={b:.2f} (true value 1)')
plt.scatter(X, y, s=12, label='data')
plt.plot(X, w*X+b, 'r', label='fit')
plt.legend(); plt.title('linear regression via gradient descent'); plt.show()

**🔗 Aurora connection**: the `gd_step(x, grad_value, lr)` implemented in this section wraps up the core update logic of gradient descent. When Month 2 brings in the training loop, this update rule reappears as `optimizer.step()`, extending the "predict → compute the error → update the weights along the gradient" logic to the parameter tensors of neural networks, and powering the full training of ASR, music generation and LLMs.


## Parameter experiment: the size of lr decides between converging and oscillating

Starting from `x=-2.0`, run 8 steps on `f(x)=(x−3)²` with `lr=0.2`, printing the value of `x` at each step:

- `lr=0.2`: a moderate step size; `x` approaches the minimum at 3 monotonically, with a clear convergence trend visible within 8 steps.

Change `lr` and watch the different behaviors: with `lr=0.9` the steps are larger and `x` overshoots the minimum at 3 every step, but the swings shrink (it still converges, since 0<lr<1); with `lr=0.01` the steps are tiny, so convergence is steady but takes many more steps. See for yourself just how much the size of the learning rate affects convergence.

In [ ]:
def f(x):
    return (x - 3)**2

# Compare the convergence behavior of four learning rates
# converges when lr<1 and |1-2*lr|<1 (i.e. 0<lr<1); at lr=1.1, |1-2*1.1|=1.2>1 and the sequence diverges
for lr in [0.01, 0.2, 0.9, 1.1]:
    x = -2.0
    for step in range(8):
        grad = 2 * (x - 3)
        x = x - lr * grad
    fx = f(x)
    status = '✅ converging' if abs(x - 3) < 5 else '❌ diverging'
    print(f'lr={lr:.2f} | after 8 steps x={x:9.3f} | f(x)={fx:10.2f} | {status}')


## 🎯 The payoff later

Today's one-line "take a step along the negative gradient" update rule, `x ← x − lr · grad`, gets used the moment **L56 (backpropagation)** has the gradient of every weight, and in **L58 (the training loop)** it becomes `optimizer.step()` — every step of training a neural network is, at heart, today's blindfolded step downhill.

## Wrap-up

You can now call `gd_step(x, grad_value, lr)` to update `x` step by step, and check the convergence on `f(x) = (x−3)²`. This corresponds to the parameter update at each step of the training loop in Month 2. The next lesson (L26) uses cviz contour plots to show gradient-descent trajectories as they converge, comparing at a glance how fast different learning rates and starting points converge.

Further on, in L58's training loop, this update rule joins up with `Value`'s gradient propagation to become one complete parameter-update step.

In [ ]:
# ── Parameter experiment: how the learning rate affects convergence ─────────────────────────────
import matplotlib.pyplot as plt

learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for lr, color in zip(learning_rates, colors):
    # minimize f(w) = (w - 3)^2, gradient = 2*(w - 3), optimum w=3
    w, losses = 0.0, []
    for _ in range(100):
        grad = 2 * (w - 3.0)
        w -= lr * grad
        losses.append((w - 3.0) ** 2)
    axes[0].plot(losses, label=f'lr={lr}', color=color)

axes[0].set_xlabel('iteration'); axes[0].set_ylabel('loss (w-3)²')
axes[0].set_title('Convergence curves for different learning rates'); axes[0].legend(); axes[0].set_yscale('log')
axes[0].set_ylim(bottom=1e-10)

# final loss after 100 steps for each learning rate
final_losses = []
for lr in learning_rates:
    w = 0.0
    for _ in range(100):
        w -= lr * 2 * (w - 3.0)
    final_losses.append((w - 3.0) ** 2)

axes[1].bar([str(lr) for lr in learning_rates], final_losses, color=colors)
axes[1].set_xlabel('learning rate'); axes[1].set_ylabel('loss after 100 steps')
axes[1].set_title('Learning rate vs final loss')
plt.tight_layout(); plt.show()

assert final_losses[2] < final_losses[0], "lr=0.1 should converge faster than lr=0.001"
print("✅ What to notice: lr too small → slow convergence; lr too large → oscillation or divergence; best lr ≈ 0.1–0.5 (depends on the problem)")

---
⬇️ **Clearing check**: you've read the wrap-up; now do the whiteboard challenge below before ticking off the self-review.


---

## 🌐 What the gradient means in many dimensions

**One-dimensional recap**: f(x) = (x−3)², and its derivative f'(x) = 2(x−3) is a scalar: the rate of change of the function along the x-axis.

**The two-dimensional case**: f(x, y) = x² + y². Now there are two directions, and we need two numbers to describe the "slope".

**Partial derivatives**: differentiate with respect to one variable, treating the others as constants
- ∂f/∂x = 2x (the rate of change of f along x)
- ∂f/∂y = 2y (the rate of change of f along y)

**The gradient vector** ∇f (nabla): line up all the partial derivatives as a vector
$$\nabla f(x, y) = \begin{bmatrix} \partial f/\partial x \\ \partial f/\partial y \end{bmatrix} = \begin{bmatrix} 2x \\ 2y \end{bmatrix}$$

At the point (x, y) = (2, 4), the gradient is ∇f(2,4) = [4, 8] — this vector points in the direction in which the function grows fastest.

**How does gradient descent take a step?** In the **opposite direction** to the gradient, i.e. each component of (x, y) decreases by its matching gradient component:
$$\begin{bmatrix} x_{\text{new}} \\ y_{\text{new}} \end{bmatrix} = \begin{bmatrix} x \\ y \end{bmatrix} - \text{lr} \cdot \nabla f(x, y)$$

For example, starting from (2, 4), with lr = 0.1 and gradient [4, 8]:
$$\begin{bmatrix} 1.6 \\ 3.2 \end{bmatrix} = \begin{bmatrix} 2 \\ 4 \end{bmatrix} - 0.1 \cdot \begin{bmatrix} 4 \\ 8 \end{bmatrix}$$

Each coordinate independently takes a small step "down the steepest slope in its own direction", and the result is a move toward the optimum (0, 0) in two-dimensional space.

💡 **Key insight**: the logic of gradient descent is exactly the same in one dimension and in many:
$$\text{params}_{\text{new}} = \text{params}_{\text{old}} - \text{lr} \times \nabla f(\text{params})$$
only now the parameters and the gradient are vectors.

## ✏️ Whiteboard challenge: gradient descent by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: f(x) = (x−3)², f'(x) = ?
Starting from x=0 with lr=0.1, compute x₁ and x₂ by hand.

**Question 2**: What should `gd_step(0.0, -6.0, 0.1)` return?

**Question 3**: f(x) = (x−3)², lr=2.0.
Starting from x=0, x₁ = ? x₂ = ?
Will this converge or diverge? (Hint: compute the ratio |1 - 2·lr|)

**Question 4**: f(x, y) = x² + y², ∇f(x,y) = ?
Starting from (2, 4) with lr=0.1, after one step (x₁, y₁) = ?

When you've finished the derivations, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: f'(x)=2(x-3), x0=0, lr=0.1
f = lambda x: (x - 3)**2
grad_f = lambda x: 2 * (x - 3)
x0, lr = 0.0, 0.1
x1_expected = x0 - lr * grad_f(x0)   # 0 - 0.1*(-6) = 0.6
x2_expected = x1_expected - lr * grad_f(x1_expected)  # 0.6 - 0.1*(-4.8) = 1.08
assert np.isclose(x1_expected, 0.6, atol=1e-10)
assert np.isclose(x2_expected, 1.08, atol=1e-10)
print(f"Q1 ✅  x₁={x1_expected:.3f}, x₂={x2_expected:.3f}  (converging toward x*=3)")

# Q2: gd_step(0.0, -6.0, 0.1) = 0 - 0.1*(-6) = 0.6
try:
    result = gd_step(0.0, -6.0, 0.1)
    assert np.isclose(result, 0.6, atol=1e-12), f"gd_step should return 0.6, got {result}"
    print(f"Q2 ✅  gd_step(0.0,-6.0,0.1) = {result}")
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement gd_step() first, then run the answer-check cell")

# Q3: lr=2.0 diverges
lr_big = 2.0
x_div = 0.0
xs_div = [x_div]
for _ in range(4):
    x_div = x_div - lr_big * grad_f(x_div)
    xs_div.append(x_div)
ratio = abs(1 - 2 * lr_big)  # = 3 > 1 → diverges
assert ratio > 1, "with lr=2 the ratio should be > 1"
print(f"Q3 ✅  lr=2.0 → ratio |1-2·lr|={ratio}")
print(f"       the sequence of x: {[round(v,1) for v in xs_div]}")
print(f"       x₁=12 (overshoots the minimum by a long way), x₂=-24 (overshoots back the other way), x₃=84......")
print(f"       a ratio > 1 means the error keeps getting multiplied by 3, so the sequence diverges")

# Q4: f(x,y)=x²+y², ∇f=(2x,2y), starting from (2,4) with lr=0.1
lr_2d = 0.1
p = np.array([2., 4.])
grad_2d = np.array([2*p[0], 2*p[1]])   # [4, 8]
p1 = p - lr_2d * grad_2d              # [2-0.4, 4-0.8] = [1.6, 3.2]
assert np.allclose(p1, [1.6, 3.2], atol=1e-12)
print(f"Q4 ✅  ∇f(2,4)=[4,8]")
print(f"       one update: ({p[0]},{p[1]}) - 0.1×[4,8] = ({p1[0]:.1f},{p1[1]:.1f})")
print(f"       each component walks against its gradient, and both move toward the optimum (0,0)")
print(f"       that is gradient descent in many dimensions: parameter vector ← parameter vector - lr × gradient vector")

print("\n🎉 Gradient-descent whiteboard challenge passed! You've internalized the basic update step of neural-network training.")

In [ ]:
# ✏️ Self-review for this lesson
l25_review = {
    "gd_update_formula":          None,  # remember x←x-lr·∇f(x)? True/False
    "gd_step_implemented":        None,  # gd_step implemented and assertions pass? True/False
    "lr_convergence_condition":   None,  # know the criterion: lr<1 converges, lr>1 diverges? True/False
    "2d_gradient_descent":        None,  # can do a component-wise GD update on a multivariable function? True/False
    "whiteboard_passed":          None,  # whiteboard challenge done on paper? True/False
}

unfilled = [k for k, v in l25_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l25_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L25 fully cleared! On to L26: calculus visualized')

---

→ **Next lesson** · [L26 · Calculus visualized](L26_visual_calculus.ipynb)

> Next lesson covers **calculus visualized**: animated demos of tangents, contour lines and gradient-descent trajectories.